### Importation des bibliothèques

In [51]:
import pandas as pd
import requests
import time
import os

### Importation des datasets

In [5]:
rating_df = pd.read_csv('collaborative_books_df.csv')
book_id_map = pd.read_csv('book_id_map.csv')
book_titles = pd.read_csv('book_titles.csv')
metadata = pd.read_csv('collaborative_book_metadata.csv')
user_id_map = pd.read_csv('user_id_map.csv')

In [31]:
metadata.head()

,book_id,title,num_pages,ratings_count,description,genre,book_id_mapping
0,5899779,Pride and Prejudice and Zombies Pride and Prej...,320,105537,The New York Times Best Seller is now a major ...,"['fantasy, paranormal', 'romance', 'fiction', ...",808
1,872333,Blue Bloods Blue Bloods 1,302,117633,"When the Mayflower set sail in 1620, it carrie...","['young-adult', 'fantasy, paranormal', 'romanc...",217
2,15507958,Me Before You Me Before You 1,369,609327,Louisa Clark is an ordinary young woman living...,"['romance', 'fiction']",385
3,66559,Sharp Objects,254,208394,"Fresh from a brief stay at a psych hospital, r...","['mystery, thriller, crime', 'fiction']",192
4,7235533,The Way of Kings The Stormlight Archive 1,1007,151473,"Speak again the ancient oaths,\nLife before de...","['fantasy, paranormal', 'fiction']",873


In [11]:
def overview_plus(df, name, n=5):
    """Affiche un aperçu complet d'un DataFrame : head, types, manquants, doublons."""
    print('\n' + '='*60)
    print(f'{name}  ->  {df.shape[0]:,} lignes, {df.shape[1]} colonnes')
    print('='*60)
    display(df.head(n))
    print('\n--- Types et non-nuls ---')
    df.info()
    print('\n--- Valeurs manquantes ---')
    miss = df.isna().sum().sort_values(ascending=False)
    miss = miss[miss > 0].head(10)
    print('Aucune valeur manquante ' if miss.empty else miss)
    print(f'\n--- Doublons : {df.duplicated().sum()} ---')

overview_plus(book_id_map,  'book_id_map')
overview_plus(book_titles,  'book_titles')
overview_plus(metadata,     'metadata (brut)')
overview_plus(rating_df,    'rating_df (brut)')
overview_plus(user_id_map,  'user_id_map')


book_id_map  ->  2,360,650 lignes, 2 colonnes


,book_id_csv,book_id
0,0,34684622
1,1,34536488
2,2,34017076
3,3,71730
4,4,30422361



--- Types et non-nuls ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2360650 entries, 0 to 2360649
Data columns (total 2 columns):
 #   Column       Dtype
---  ------       -----
 0   book_id_csv  int64
 1   book_id      int64
dtypes: int64(2)
memory usage: 36.0 MB

--- Valeurs manquantes ---
Aucune valeur manquante 

--- Doublons : 0 ---

book_titles  ->  1,447,341 lignes, 2 colonnes


,title,book_id
0,The Unschooled Wizard Sun Wolf and Starhawk 12,7327624
1,Best Friends Forever,6066819
2,The Aeneid for Boys and Girls,287141
3,Alls Fairy in Love and War Avalon Web of Magic 8,6066812
4,The Devils Notebook,287149



--- Types et non-nuls ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1447341 entries, 0 to 1447340
Data columns (total 2 columns):
 #   Column   Non-Null Count    Dtype 
---  ------   --------------    ----- 
 0   title    1437948 non-null  object
 1   book_id  1447341 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 22.1+ MB

--- Valeurs manquantes ---
title    9393
dtype: int64

--- Doublons : 0 ---

metadata (brut)  ->  96 lignes, 7 colonnes


...


--- Types et non-nuls ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   book_id          96 non-null     int64 
 1   title            96 non-null     object
 2   num_pages        96 non-null     int64 
 3   ratings_count    96 non-null     int64 
 4   description      96 non-null     object
 5   genre            96 non-null     object
 6   book_id_mapping  96 non-null     int64 
dtypes: int64(4), object(3)
memory usage: 5.4+ KB

--- Valeurs manquantes ---
Aucune valeur manquante 

--- Doublons : 0 ---

rating_df (brut)  ->  196,296 lignes, 6 colonnes


,title,book_id,user_id_mapping,book_id_mapping,pred_rating,rating
0,I Am the Messenger,19057,1537,299,4.5,5.0
1,I Am the Messenger,19057,23039,299,4.9,3.0
2,I Am the Messenger,19057,39096,299,3.9,3.0
3,I Am the Messenger,19057,14631,299,4.7,4.0
4,I Am the Messenger,19057,32816,299,4.3,5.0



--- Types et non-nuls ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 196296 entries, 0 to 196295
Data columns (total 6 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   title            196296 non-null  object 
 1   book_id          196296 non-null  int64  
 2   user_id_mapping  196296 non-null  int64  
 3   book_id_mapping  196296 non-null  int64  
 4   pred_rating      196296 non-null  float64
 5   rating           196296 non-null  float64
dtypes: float64(2), int64(3), object(1)
memory usage: 9.0+ MB

--- Valeurs manquantes ---
Aucune valeur manquante 

--- Doublons : 0 ---

user_id_map  ->  876,145 lignes, 2 colonnes


,user_id_csv,user_id
0,0,8842281e1d1347389f2ab93d60773d4d
1,1,72fb0d0087d28c832f15776b0d936598
2,2,ab2923b738ea3082f5f3efcbbfacb218
3,3,d986f354a045ffb91234e4af4d1b12fd
4,4,7504b2aee1ecb5b2872d3da381c6c91e



--- Types et non-nuls ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 876145 entries, 0 to 876144
Data columns (total 2 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   user_id_csv  876145 non-null  int64 
 1   user_id      876145 non-null  object
dtypes: int64(1), object(1)
memory usage: 13.4+ MB

--- Valeurs manquantes ---
Aucune valeur manquante 

--- Doublons : 0 ---


In [7]:
# Supprimer l'index parasite et renommer les colonnes
if 'Unnamed: 0' in rating_df.columns:
    rating_df = rating_df.drop(columns=['Unnamed: 0'])

rating_df = rating_df.rename(columns={
    'Predicted Rating': 'pred_rating',
    'Actual Rating':    'rating'
})
rating_df['rating'] = rating_df['rating'].astype(float)

print(f'Doublons       : {rating_df.duplicated().sum()}')
print(f'Valeurs manq.  : {rating_df.isna().sum().sum()}')
print(f'Shape finale   : {rating_df.shape}')
display(rating_df.head())

Doublons       : 0
Valeurs manq.  : 0
Shape finale   : (196296, 6)


,title,book_id,user_id_mapping,book_id_mapping,pred_rating,rating
0,I Am the Messenger,19057,1537,299,4.5,5.0
1,I Am the Messenger,19057,23039,299,4.9,3.0
2,I Am the Messenger,19057,39096,299,3.9,3.0
3,I Am the Messenger,19057,14631,299,4.7,4.0
4,I Am the Messenger,19057,32816,299,4.3,5.0


In [9]:
# Supprimer les colonnes inutiles
cols_to_drop = [c for c in ['Unnamed: 0', 'image_url', 'url', 'name'] if c in metadata.columns]
metadata = metadata.drop(columns=cols_to_drop)

# Normaliser les colonnes textuelles
for col in ['title', 'description', 'genre']:
    metadata[col] = metadata[col].astype(str).str.strip()

print(f'Shape metadata nettoyé : {metadata.shape}')
display(metadata.head())

Shape metadata nettoyé : (96, 7)


,book_id,title,num_pages,ratings_count,description,genre,book_id_mapping
0,5899779,Pride and Prejudice and Zombies Pride and Prej...,320,105537,The New York Times Best Seller is now a major ...,"['fantasy, paranormal', 'romance', 'fiction', ...",808
1,872333,Blue Bloods Blue Bloods 1,302,117633,"When the Mayflower set sail in 1620, it carrie...","['young-adult', 'fantasy, paranormal', 'romanc...",217
2,15507958,Me Before You Me Before You 1,369,609327,Louisa Clark is an ordinary young woman living...,"['romance', 'fiction']",385
3,66559,Sharp Objects,254,208394,"Fresh from a brief stay at a psych hospital, r...","['mystery, thriller, crime', 'fiction']",192
4,7235533,The Way of Kings The Stormlight Archive 1,1007,151473,"Speak again the ancient oaths,\nLife before de...","['fantasy, paranormal', 'fiction']",873


In [13]:
book_titles = book_titles.dropna(subset=['title'])
book_titles['title'] = book_titles['title'].str.strip()
book_titles = book_titles[book_titles['title'] != '']

print(f'Valeurs manquantes après nettoyage : {book_titles.isna().sum().to_dict()}')
print(f'Shape : {book_titles.shape}')

Valeurs manquantes après nettoyage : {'title': 0, 'book_id': 0}
Shape : (1384641, 2)


In [15]:
# Livres avec metadata
ids_metadata = set(metadata['book_id'])

# Tous les livres
ids_all = set(book_titles['book_id'])

# Livres sans metadata
ids_missing = ids_all - ids_metadata

print(f"Nombre de livres sans metadata: {len(ids_missing)}")

Nombre de livres sans metadata: 1384545


In [17]:
livres_sans_metadata = book_titles[
    book_titles['book_id'].isin(ids_missing)
]

livres_sans_metadata.head()

,title,book_id
0,The Unschooled Wizard Sun Wolf and Starhawk 12,7327624
1,Best Friends Forever,6066819
2,The Aeneid for Boys and Girls,287141
3,Alls Fairy in Love and War Avalon Web of Magic 8,6066812
4,The Devils Notebook,287149


In [35]:
def fetch_google_books_full(title):
    url = f"https://www.googleapis.com/books/v1/volumes?q=intitle:{title}"
    
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        
        if "items" in data:
            volume = data["items"][0]["volumeInfo"]
            
            description = volume.get("description", None)
            categories = volume.get("categories", [])
            genre = ", ".join(categories)
            
            num_pages = volume.get("pageCount", None)
            
            # ratings_count n’est pas toujours dispo
            ratings_count = volume.get("ratingsCount", None)
            
            return {
                "description": description,
                "genre": genre,
                "num_pages": num_pages,
                "ratings_count": ratings_count
            }
        
    except:
        return None

    return None

In [37]:
def fetch_open_library_full(title):
    url = f"https://openlibrary.org/search.json?title={title}"
    
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        
        if data["docs"]:
            doc = data["docs"][0]
            
            subjects = doc.get("subject", [])
            genre = ", ".join(subjects[:3])
            
            num_pages = doc.get("number_of_pages_median", None)
            
            return {
                "description": None,  # souvent absent
                "genre": genre,
                "num_pages": num_pages,
                "ratings_count": None
            }
        
    except:
        return None

    return None

In [39]:
def fetch_metadata_full(title):
    
    # Google Books d’abord
    result = fetch_google_books_full(title)
    
    if result and (result["description"] or result["genre"]):
        return result
    
    # fallback Open Library
    return fetch_open_library_full(title)

In [41]:
sample = livres_sans_metadata.sample(10, random_state=42)

results = []

for _, row in sample.iterrows():
    data = fetch_metadata_full(row["title"])
    
    if data:
        data["title"] = row["title"]
        results.append(data)

df_enriched = pd.DataFrame(results)
df_enriched

,description,genre,num_pages,ratings_count,title
0,"Non ingombrare, non essere ingombranti: è l’un...",Biography & Autobiography,85.0,None,Solo bagaglio a mano
1,Kathryn Lane thinks she has life figured out: ...,,402.0,None,After the Sky Fell Down
2,None,Fiction,194.0,None,A Daughters a Daughter
3,None,,NaN,None,My Brothers Keeper
4,None,,NaN,None,Castillos de cartn
5,Dicono che sia un condottiero spietato e infal...,Fiction,397.0,None,Il battito delle sue ali
6,Some say every inch of this earth has been cov...,Armenia (Republic),176.0,None,Tour de Armenia
7,Describes how the thinking of the boy Jesus Ch...,,64.0,None,Young Jesus Asks Questions


In [43]:
df_enriched["has_metadata"] = (
    df_enriched["description"].notna() & (df_enriched["description"] != "")
) | (
    df_enriched["genre"].notna() & (df_enriched["genre"] != "")
)

In [45]:
nb_avec = df_enriched["has_metadata"].sum()
nb_sans = len(df_enriched) - nb_avec

print(f"Avec metadata: {nb_avec}")
print(f"Sans metadata: {nb_sans}")

Avec metadata: 6
Sans metadata: 2


In [57]:
output_file = "enriched_full.csv"

# Si le fichier existe déjà → reprendre
if os.path.exists(output_file):
    df_existing = pd.read_csv(output_file)
    processed_titles = set(df_existing["title"])
else:
    df_existing = pd.DataFrame()
    processed_titles = set()

In [ ]:
results = []

for i, (_, row) in enumerate(livres_sans_metadata.iterrows()):
    
    title = row["title"]
    
    # Skip déjà fait
    if title in processed_titles:
        continue
    
    data = fetch_metadata_full(title)
    
    if data:
        data["title"] = title
        results.append(data)
    
    # Sauvegarde toutes les 20 itérations
    if i % 20 == 0:
        df_temp = pd.DataFrame(results)
        df_temp.to_csv(output_file, mode='a', header=not os.path.exists(output_file), index=False)
        results = []
        print(f"{i} livres traités et sauvegardés")
    
    # Pause API
    time.sleep(0.2)

0 livres traités et sauvegardés
20 livres traités et sauvegardés
40 livres traités et sauvegardés
60 livres traités et sauvegardés
80 livres traités et sauvegardés
100 livres traités et sauvegardés
120 livres traités et sauvegardés
140 livres traités et sauvegardés
160 livres traités et sauvegardés
180 livres traités et sauvegardés
200 livres traités et sauvegardés
220 livres traités et sauvegardés
240 livres traités et sauvegardés
260 livres traités et sauvegardés
280 livres traités et sauvegardés
300 livres traités et sauvegardés
320 livres traités et sauvegardés
340 livres traités et sauvegardés
360 livres traités et sauvegardés
380 livres traités et sauvegardés
400 livres traités et sauvegardés
420 livres traités et sauvegardés
440 livres traités et sauvegardés
460 livres traités et sauvegardés
480 livres traités et sauvegardés
500 livres traités et sauvegardés
520 livres traités et sauvegardés
540 livres traités et sauvegardés
560 livres traités et sauvegardés
580 livres traités e